In [1]:
%matplotlib inline

import os, random, time, sys, warnings
from timeit import default_timer as timer
from datetime import timedelta
from datetime import datetime

import numpy as np
import scipy as sp
from scipy.spatial import distance
from scipy import signal
import pandas as pd
from tqdm import tqdm
import bct

import neurogym as ngym
import torch
import torch.nn as nn

from src.neural_network import RNN, run_testing
from src.utils import normalize_x, build_reg_ken, fix_labels, get_weight_masks, get_weight_masks_schaefer, get_file_str

# import plotting libraries
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 10})
plt.rcParams["svg.fonttype"] = "none"
plt.rc('font', family='DejaVu Sans')
import seaborn as sns
sns.set_style("white")
from src.plotting import my_reg_plot

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
save_figs = True

In [3]:
# directories
datadir = '/home/lindenmp/research_projects/neuro_rnn/data'
modeldir = '/media/lindenmp/storage_ssd/research_projects/neuro_rnn/results/pytorch/model'
outdir = '/home/lindenmp/research_projects/neuro_rnn/results/figs'

# data parameters
task = 'PerceptualDecisionMaking-v0'
# task = 'MultiSensoryIntegration-v0'
# task = 'ContextDecisionMaking-v0'
dt = 100
batch_size = 32
decision = 400
if task == 'PerceptualDecisionMaking-v0':
    seq_len = 22
elif task == 'MultiSensoryIntegration-v0':
    seq_len = 11
elif task == 'ContextDecisionMaking-v0':
    seq_len = 13
seq_len = seq_len + int((decision - 100) / dt)
seq_len_multi = 5
seq_len = seq_len * seq_len_multi
print(seq_len)

# RNN model and training parameters
rnn_model = 'rnn-tanh'
hidden_size = 100
n_runs = 25
n_epochs = 30000
lr = 0.001

# regularization parameters
reg_type = 'l2'
reg_weight = 0.002
mask_weights = True
kernel_type = 'sa_axis'
# kernel_type = 'euclidean'
# kernel_type = None

125


In [4]:
timing = {'fixation': 200, 'stimulus': 1000, 'delay': 0, 'decision': decision}
env_kwargs = {'dt': dt, 'timing': timing}

config = {
    'datadir': datadir, 'outdir': outdir,
    'task': task, 'dt': dt, 'seq_len': seq_len, 'batch_size': batch_size,  # data parameters
    'rnn_model': rnn_model, 'hidden_size': hidden_size, 'n_runs': n_runs, 'n_epochs': n_epochs, 'lr': lr, 'mask_weights': mask_weights,  # RNN model and training parameters
    'reg_type': reg_type, 'reg_weight': reg_weight, 'kernel_type': kernel_type, # regularization parameters
    'env_kwargs': env_kwargs,
}

In [5]:
# weight masks
if mask_weights:
    centroids = pd.read_csv(os.path.join(datadir, 'schaefer{0}_centroids.csv'.format(hidden_size * 2)))
    centroids = centroids[:hidden_size]
    roi_names = list(centroids['ROI Name'])
    input_system = 'Vis'
    output_system = 'Default'
    masks = get_weight_masks_schaefer(roi_names=roi_names, input_system=input_system, output_system=output_system)
    n_io = '{0}-{1}'.format(np.sum(masks['input_weight_mask']), np.sum(masks['output_weight_mask']))
else:
    n_io = 'na'
config['n_io'] = n_io

file_str = get_file_str(config)
print(file_str)

task-PerceptualDecisionMaking-v0-125-400_model-rnn-tanh-100-32-0.001-25-30000_wmask-True-14-27_reg-l2-0.002-sa_axis


In [6]:
# setup weight masks
if mask_weights:
    input_weight_mask = np.zeros((hidden_size,))
    output_weight_mask = np.zeros((hidden_size,))
else:
    input_weight_mask = None
    output_weight_mask = None

In [7]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = 'cpu'
print(device)

# load a model to get epochs
checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)
run = 0
epoch_list = []
N = 10
for idx in range(0, len(list(checkpoint[run].keys())), N):
    epoch_list.append(list(checkpoint[run].keys())[idx])
n_logged_epochs = len(epoch_list)
del checkpoint
print(n_logged_epochs)
print(epoch_list)

cuda
31
[0, 999, 1999, 2999, 3999, 4999, 5999, 6999, 7999, 8999, 9999, 10999, 11999, 12999, 13999, 14999, 15999, 16999, 17999, 18999, 19999, 20999, 21999, 22999, 23999, 24999, 25999, 26999, 27999, 28999, 29999]


In [8]:
mask_labels = ['input_output', 'output_input',
               'input_bystanders', 'bystanders_input',
               'output_bystanders', 'bystanders_output']
n_masks = len(mask_labels)
mask_list = []
for i in np.arange(n_masks):
    mask_list.append(masks[mask_labels[i]])
print(n_masks)

6


In [9]:
# number of trials for lesion analysis
n_trials = 100

## Lesion input, output, and internal edges

In [10]:
run_cell = False

In [11]:
if run_cell:
    # for decision in [100, 200, 300, 400, 500, 600, 700, 1100]:
    for decision in [400,]:
        if task == 'PerceptualDecisionMaking-v0':
            seq_len = 22
        elif task == 'MultiSensoryIntegration-v0':
            seq_len = 11
        elif task == 'ContextDecisionMaking-v0':
            seq_len = 13
        seq_len = seq_len + int((decision - 100) / dt)
        seq_len = seq_len * seq_len_multi

        timing = {'decision': decision}
        env_kwargs = {'dt': dt, 'timing': timing}
        config['env_kwargs'] = env_kwargs

        # setup dataset
        dataset = ngym.Dataset(config['task'], env_kwargs=env_kwargs, batch_size=config['batch_size'], seq_len=config['seq_len'])
        input_size = dataset.env.observation_space.shape[0]
        num_classes = dataset.env.action_space.n

        for reg_type in ['l2',]:
            config['reg_type'] = reg_type

            for reg_weight in [0.002,]:
                config['reg_weight'] = reg_weight

                for kernel_type in ['sa_axis', 'euclidean']:
                    config['kernel_type'] = kernel_type
                    file_str = get_file_str(config)
                    print(file_str)

                    if os.path.isfile(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'lesioned-accuracy'))):
                        pass
                    else:
                        # load model checkpoint
                        checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)

                        # setup kernel
                        if kernel_type is None:
                            regularization_kernel = None
                        else:
                            regularization_kernel = np.zeros((hidden_size, hidden_size))

                        # setup model
                        model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
                        type=rnn_model, regularization_kernel=regularization_kernel,
                        input_weight_mask=input_weight_mask, output_weight_mask=output_weight_mask).to(device)
                        model.eval()

                        # compute accuracy and lesioned accuracy
                        lesioned_accuracy = np.zeros((n_runs, n_logged_epochs, n_masks))
                        accuracy = np.zeros((n_runs, n_logged_epochs))
                        for run in tqdm(np.arange(n_runs)):
                            for i, epoch in enumerate(epoch_list):
                                model.load_state_dict(checkpoint[run][epoch])
                                accuracy[run, i], _, _, _, _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)

                                for j, mask in enumerate(mask_list):
                                    model.load_state_dict(checkpoint[run][epoch])
                                    with torch.no_grad():
                                        model.rnn.weight_hh_l0[mask] = 0
                                    lesioned_accuracy[run, i, j], _, _, _, _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)

                        # plot
                        f, ax = plt.subplots(2, 1, figsize=(9, 6))
                        x_step = int(n_epochs / (n_logged_epochs-1))
                        color_palette = sns.color_palette("Set2")
                        ax[0].set_ylim([-10, 100])
                        ax[0].set_xlabel('Epochs')
                        ax[0].set_ylabel('Test Accuracy (%)')
                        ax[1].set_ylim([-100, 10])
                        ax[1].set_xlabel('Epochs')
                        ax[1].set_ylabel('Test Accuracy (delta, %)')
                        for i in np.arange(n_masks):
                            jitter = i
                            ax[0].plot(np.arange(0, n_epochs+x_step, x_step), (np.median(lesioned_accuracy, axis=0)[:, i] * 100) - jitter, color=color_palette[i], label=mask_labels[i])
                            ax[1].plot(np.arange(0, n_epochs+x_step, x_step), ((np.median(lesioned_accuracy, axis=0)[:, i] - np.median(accuracy, axis=0)) * 100) - jitter, color=color_palette[i], label=mask_labels[i])
                        ax[0].plot(np.arange(0, n_epochs+x_step, x_step), np.median(accuracy, axis=0) * 100, color='k', label='unlesioned')
                        ax[0].legend(bbox_to_anchor = (1, 1))
                        sns.despine(offset=10, trim=True, left=False, right=True, top=True, bottom=False)

                        f.suptitle(file_str)
                        f.tight_layout()
                        plt.show()
                        f.savefig(os.path.join(outdir, '{0}_{1}.png'.format(file_str, 'lesioned-accuracy')), dpi=300, bbox_inches='tight', pad_inches=0.01)

## Lesion random edges

In [12]:
run_cell = False

In [13]:
if run_cell:
    tasks = ['PerceptualDecisionMaking-v0', 'MultiSensoryIntegration-v0', 'ContextDecisionMaking-v0']
    # tasks = ['PerceptualDecisionMaking-v0', 'MultiSensoryIntegration-v0']
    n_tasks = len(tasks)

    kernel_types = ['sa_axis', 'euclidean', None]
    kernel_labels = ['RNN-SA', 'RNN-E', 'RNN-Standard']
    n_kernels = len(kernel_types)

    n_edges = hidden_size * hidden_size
    lesion_percentages = np.linspace(0.01, 0.10, 10)
    n_lesion_perc = len(lesion_percentages)
    print(lesion_percentages)

In [14]:
if run_cell:
    lesioned_accuracy = np.zeros((n_tasks, n_kernels, n_runs, n_lesion_perc))
    for i, task in enumerate(tasks):
        if task == 'PerceptualDecisionMaking-v0':
            seq_len = 22
        elif task == 'MultiSensoryIntegration-v0':
            seq_len = 11
        elif task == 'ContextDecisionMaking-v0':
            seq_len = 13
        seq_len = seq_len + int((decision - 100) / dt)
        seq_len = seq_len * seq_len_multi

        config['task'] = task
        config['seq_len'] = seq_len

        # setup dataset
        dataset = ngym.Dataset(config['task'], env_kwargs=env_kwargs, batch_size=config['batch_size'], seq_len=config['seq_len'])
        input_size = dataset.env.observation_space.shape[0]
        num_classes = dataset.env.action_space.n
        print(input_size, num_classes)

        for j, kernel_type in enumerate(kernel_types):
            config['kernel_type'] = kernel_type
            file_str = get_file_str(config)
            print(file_str)

            # load model checkpoint
            checkpoint = torch.load(os.path.join(modeldir, file_str + '.pt'), map_location=device)

            # setup kernel
            if kernel_type is None:
                regularization_kernel = None
            else:
                regularization_kernel = np.zeros((hidden_size, hidden_size))

            # setup model
            model = RNN(input_size=input_size, hidden_size=hidden_size, num_classes=num_classes,
            type=rnn_model, regularization_kernel=regularization_kernel,
            input_weight_mask=input_weight_mask, output_weight_mask=output_weight_mask).to(device)
            model.eval()

            epoch = epoch_list[-1]
            # epoch = 14999
            for run in tqdm(np.arange(n_runs)):
                for l, lesion_perc in enumerate(lesion_percentages):
                    np.random.seed(42)
                    mask = np.zeros((hidden_size, hidden_size)).astype(bool)
                    mask = mask.reshape(n_edges)
                    indices = np.random.choice(np.arange(mask.size), replace=False, size=int(n_edges * lesion_perc))
                    mask[indices] = True
                    mask = mask.reshape(hidden_size, hidden_size)

                    model.load_state_dict(checkpoint[run][epoch])
                    with torch.no_grad():
                        model.rnn.weight_hh_l0[mask] = 0
                    lesioned_accuracy[i, j, run, l], _, _, _, _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)
            del checkpoint

In [15]:
if run_cell:
    fig_width = 8.5
    fig_height = 6
    f, ax = plt.subplots(n_tasks, n_kernels, figsize=(fig_width, fig_height))

    for i, task in enumerate(tasks):
        for j, kernel_type in enumerate(kernel_types):
            df = pd.DataFrame(data=lesioned_accuracy[i, j]*100, columns=np.round(lesion_percentages*100).astype(int))
            sns.barplot(df, ax=ax[i, j], estimator='mean', errorbar='ci', palette='rocket')

            sns.despine(offset=3, trim=False, left=False, right=True, top=True, bottom=False, ax=ax[i, j])
            ax[i, j].set_xlabel('Lesioned Edged (%)')
            ax[i, j].set_ylabel('Accuracy (%)')
            ax[i, j].set_ylim([0, 100])
            if j == 1:
                ax[i, j].set_title('{0}\n{1}'.format(task, kernel_type))
            else:
                ax[i, j].set_title('\n{0}'.format(kernel_type))

    f.tight_layout()
    plt.show()

### Single node lesioning

In [16]:
run_cell = False

In [17]:
if run_cell:
    epoch = epoch_list[-1]

    # compute accuracy and lesioned accuracy
    lesioned_accuracy = np.zeros((n_runs, hidden_size))
    accuracy = np.zeros((n_runs, hidden_size))

    for run in tqdm(np.arange(n_runs)):
        for node in np.arange(hidden_size):
            model.load_state_dict(checkpoint[run][epoch])
            with torch.no_grad():
                model.rnn.weight_hh_l0[node, :] = 0
                model.rnn.weight_hh_l0[:, node] = 0
            lesioned_accuracy[run, node], _, _ = run_testing(dataset=dataset, model=model, n_trials=n_trials, verbose=False)

In [18]:
if run_cell:
    f, ax = plt.subplots(1, 1, figsize=(5, 5))
    sns.histplot(lesioned_accuracy[:, :].mean(axis=0), ax=ax)

In [19]:
if run_cell:
    run = 20
    A = checkpoint[run][epoch]['rnn.weight_hh_l0'].detach().cpu().numpy().copy()
    A = np.abs(A)
    A = A / np.linalg.norm(A)
    thresh1 = np.quantile(A, q=0.9)
    mask = A > thresh1
    A[mask] = 1
    A[~mask] = 0
    degree = np.sum(A, axis=0)

    f, ax = plt.subplots(1, 1, figsize=(5, 5))
    my_reg_plot(degree, lesioned_accuracy[run, :], 'Binary degree', 'Lesioned Accuracy', ax, fontsize=12, annotate='both')

In [20]:
if run_cell:
    network_stats = np.zeros((n_runs,))

    for run in tqdm(np.arange(n_runs)):

        A = checkpoint[run][epoch]['rnn.weight_hh_l0'].detach().cpu().numpy().copy()
        A = np.abs(A)
        A = A / np.linalg.norm(A)
        thresh1 = np.quantile(A, q=0.9)
        mask = A > thresh1
        A[mask] = 1
        A[~mask] = 0
        degree = np.sum(A, axis=0)

        network_stats[run] = sp.stats.spearmanr(lesioned_accuracy[run, :], degree)[0]

    f, ax = plt.subplots(1, 1, figsize=(5, 5))
    sns.histplot(network_stats, ax=ax)